# House Price Prediction in Tehran

## Importing Libraries and CSV File

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("Tehran House Price.csv")

## Getting The Idea of Dataset

In [ ]:
print(f"number of rows(listings): {df.shape[0]}")
print(f"number of columns(features): {df.shape[1]}")

In [ ]:
display(df.head(20))

In [ ]:
df.info()

## Determining Missing Values

In [ ]:
missing_data = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df) ) * 100

missing_table = pd.DataFrame({
    "number of Missings": missing_data,
    "Percent of Missing": missing_percent.round(2)})
display(missing_table[missing_table["number of Missings"] > 0])

##  Describing Numbers

In [ ]:
display(df.describe().T)

## Price Distribution

In [ ]:
price_column = "Price(USD)"

if price_column in df.columns:
    plt.figure(figsize = (9,4))
    sns.histplot(df[price_column], kde=True, color='darkblue')
    skewness = df[price_column].skew()
    plt.title(f"Price Distribution (Skewness Score; {skewness:.2f})")
    plt.xlabel("Price")
    plt.ylabel("Count")
    plt.grid(alpha=0.3)
    plt.show()

    if skewness > 1:
        print(f"Right Skewed {skewness:.2f}")
        

## Scikit Tools

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

## Cleaning Area and Price Columns

In [ ]:
df['Area'] = df['Area'].astype(str).str.replace(',', '', regex=False).str.strip()
df['Area'] = pd.to_numeric(df['Area'], errors='coerce')

In [ ]:
valid_area = (df['Area'] >= 10) & (df['Area'] <= 1000)
valid_price = (df['Price'] > 0) & (df['Price'] <= df['Price'].quantile(0.99))

df_clean = df[valid_area & valid_price].copy()

### Converting Booleans 

In [ ]:
for col in ['Parking', 'Warehouse', 'Elevator']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(int)

In [ ]:
display(df_clean.head())

In [ ]:
print( f"org rows: {df.shape[0]}")
print( f"clean rows: {df_clean.shape[0]}")

## Converting df_clean into train and test split

In [ ]:
X = df_clean.drop(columns = ['Price','Price(USD)'], errors='ignore')
y_raw = df_clean['Price']
y = np.log1p(y_raw)

#Separating Numerical and Categorical Features
num_features = ['Area','Room','Parking','Warehouse','Elevator']
cat_features = ['Address']

## Building Transformers and Pipelines

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output= False ))])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_features),
    ('cat',categorical_transformer, cat_features)])

## Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing & Split Done")
print(f"X_train original shape:  {X_train.shape}")
print(f"X_train processed shape: {X_train_processed.shape}")
print(f"X_test original shape:   {X_test.shape}")
print(f"X_test processed shape:  {X_test_processed.shape}")

## ML Algorithms Tools

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

### Defining models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'K-Nearest Neighbors': KNeighborsRegressor(n_neighbors=5),
    'Random Forest': RandomForestRegressor(n_estimators = 100, random_state=42)}

## Training Models

In [ ]:
results = []
best_model_name = None
best_r2_score = -float('inf')
best_pipeline = None

for name, model in models.items():
    full_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)])

    full_pipeline.fit(X_train, y_train)

    y_pred_log = full_pipeline.predict(X_test)

    y_test_real = np.expm1(y_test)
    y_pred_real = np.expm1(y_pred_log)

    r2 = r2_score(y_test, y_pred_log)
    mae = mean_absolute_error(y_test_real, y_pred_real)
    rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))

    results.append({
        'Model': name,
        'R2 Score': round(r2,4),
        'MAE (Tomans)': round(mae, 2),
        'RMSE (Tomans)': round(rmse, 2)})

    if r2 > best_r2_score:
        best_r2_score = r2
        best_model_name = name
        best_pipeline = full_pipeline

In [ ]:
results_df = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False)
print("MODEL EVAL RESULTS")
display(results_df)
print(f"best model is: {best_model_name}")

## Importing Tools

In [ ]:
from sklearn.model_selection import GridSearchCV

## Defining Pipeline and Search Grid

In [ ]:
base_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(random_state = 42))])

param_grid = {
    'regressor__alpha': [0.01, 0.1, 0.5, 5.0, 10.0, 50.0, 100.0, 500.0],
    'regressor__solver': ['auto', 'svd', 'cholesky', 'lsqr', 'saga']}

## Running GridSearchCV with 5-fold cross_validation

In [ ]:

print("Optimizing Ridge Regression alpha via 5-Fold Cross-Validation")

grid_search = GridSearchCV(
    estimator = base_pipeline,
    param_grid = param_grid,
    cv = 5,
    scoring = 'r2',
    n_jobs = -1,
    verbose = 1)

grid_search.fit(X_train, y_train)

best_pipeline = grid_search.best_estimator_

print("\n Grid Search Done")
print("Best Hyperparameters Found:")
for param, val in grid_search.best_params_.items():
    print(f" . {param}: {val}")

#Evaluate Tuned Model on Test Set
y_pred_log = best_pipeline.predict(X_test)

y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred_log)

final_r2 = r2_score(y_test, y_pred_log)
final_mae = mean_absolute_error(y_test_real, y_pred_real)
final_rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))

print("\nFINAL TUNED RIDGE MODEL TEST RESULTS")
print(f"R² Score:     {round(final_r2, 4)}")
print(f"MAE (Tomans): {final_mae:,.2f}")
print(f"RMSE (Tomans):{final_rmse:,.2f}")                 

In [ ]:
def predict_house_price(sample_dict):
    df_sample = pd.DataFrame([sample_dict])
    
    log_prediction = best_pipeline.predict(df_sample)
    predicted_price = np.expm1(log_prediction)[0]
    
    return predicted_price
#Test Inference Function 
sample_house = {
    'Area': 95,
    'Room': 1,
    'Parking': 1,
    'Warehouse': 1,
    'Elevator': 1,
    'Address': 'Saadat Abad'
}

predicted_price = predict_house_price(sample_house)
print("Sample Inference Test:")
print(f"Input Features: {sample_house}")
print(f"Estimated Price: {predicted_price:,.0f} Tomans")